# Persistence in LangGraph — Production Pattern

This notebook simulates a real production scenario:
- A workflow runs and **fails mid-way** (in `explain_joke` node)
- The checkpoint saves the state **up to the last successful node**
- On retry, the program **detects the existing checkpoint** and resumes — no re-running of completed nodes

## 1. Imports & Setup

In [1]:
from typing import TypedDict
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import InMemorySaver
from pprint import pprint

load_dotenv()

True

## 2. LLM

In [2]:
llm = ChatGroq(model="qwen/qwen3-32b", temperature=0.4, reasoning_format="hidden")

## 3. State

In [3]:
class JokeState(TypedDict):
    topic: str
    joke: str
    explanation: str

## 4. Nodes

`explain_joke` has a simulated failure on the **first call**.

In production this could be: API timeout, rate limit, DB write failure, etc.

In [5]:
# Tracks how many times explain_joke has been called
explain_call_count = 0

def generate_joke   (state: JokeState) -> JokeState:
    print("[Node] generate_joke: running...")
    response = llm.invoke(
        f"Generate a funny and original joke on the topic: {state['topic']}. Reply with only the joke."
    )
    print("[Node] generate_joke: done")
    return {"joke": response.content}


def explain_joke(state: JokeState) -> JokeState:
    global explain_call_count
    explain_call_count += 1

    print(f"[Node] explain_joke: attempt #{explain_call_count}")

    # Simulate a failure on the first attempt
    if explain_call_count == 1:
        raise RuntimeError("[Simulated] explain_joke failed — e.g. API timeout or rate limit")

    response = llm.invoke(
        f"Explain the joke: {state['joke']}. Reply with only the joke explanation."
    )
    print("[Node] explain_joke: done")
    return {"explanation": response.content}

## 5. Build Graph

In [6]:
builder = StateGraph(JokeState)

builder.add_node("generate_joke", generate_joke)
builder.add_node("explain_joke", explain_joke)

builder.add_edge(START, "generate_joke")
builder.add_edge("generate_joke", "explain_joke")
builder.add_edge("explain_joke", END)

## 6. Compile with Checkpointer

In [7]:
checkpointer = InMemorySaver()
workflow = builder.compile(checkpointer=checkpointer)

## 7. Production Runner

The caller **always calls with the same args** — topic included, every time.

The runner checks `next` internally:
- `next` has nodes → unfinished work → resume with `None`
- `next` is empty + values exist → already completed → return saved result
- No checkpoint at all → start fresh

In [9]:
def run_workflow(thread_id: str, topic: str):
    config = {"configurable": {"thread_id": thread_id}}

    existing_state = workflow.get_state(config)

    if existing_state.values and existing_state.next:
        # Checkpoint exists + unfinished nodes remain → resume
        print(f"\n[Runner] Resuming from step {existing_state.metadata.get('step')}. Next: {existing_state.next}\n")
        result = workflow.invoke(input=None, config=config)

    elif existing_state.values and not existing_state.next:
        # Checkpoint exists + all nodes done → already completed
        print("\n[Runner] Workflow already completed. Returning saved result.\n")
        result = existing_state.values

    else:
        # No checkpoint at all → fresh start
        print(f"\n[Runner] No checkpoint found. Starting fresh with topic='{topic}'...\n")
        result = workflow.invoke(input={"topic": topic}, config=config)

    return result, config

## 8. Demo — First Run (will fail at `explain_joke`)

In [10]:
thread_id = "job_001"

try:
    result, config = run_workflow(thread_id=thread_id, topic="robots")
except RuntimeError as e:
    print(f"\n[Error] Workflow failed: {e}")
    print("[Error] State has been saved up to the last successful node.")


[Runner] No checkpoint found. Starting fresh with topic='robots'...

[Node] generate_joke: running...
[Node] generate_joke: done
[Node] explain_joke: attempt #1

[Error] Workflow failed: [Simulated] explain_joke failed — e.g. API timeout or rate limit
[Error] State has been saved up to the last successful node.


## 9. Inspect Saved State After Failure

Even though the workflow crashed, LangGraph saved the checkpoint after `generate_joke` completed.

Notice `next` shows `explain_joke` — that's where it will resume.

In [11]:
config = {"configurable": {"thread_id": thread_id}}
snapshot = workflow.get_state(config)

print("Step:  ", snapshot.metadata.get("step"))
print("Next:  ", snapshot.next)       # Should show ('explain_joke',)
print("Values:", snapshot.values)     # Should have topic + joke, but NO explanation yet

Step:   1
Next:   ('explain_joke',)
Values: {'topic': 'robots', 'joke': 'Why did the robot get kicked out of the bakery? Because it kept making bread rolls—turns out, the Three Laws of Robotics don’t cover pastry dough! 🍞🤖'}


## 10. Demo — Restart (exact same call as first run)

In production, the crashed job is simply **restarted with the same parameters**.  
No special resume call. No `None`. Just the same function, same args.

In [12]:
try:
    result, config = run_workflow(thread_id=thread_id, topic="robots")  # Same call as before
    print("\n[Success] Workflow completed!")
except RuntimeError as e:
    print(f"\n[Error] {e}")


[Runner] Resuming from step 1. Next: ('explain_joke',)

[Node] explain_joke: attempt #2
[Node] explain_joke: done

[Success] Workflow completed!


## 11. Final Output

In [13]:
final_snapshot = workflow.get_state(config)

print("Topic:\n", final_snapshot.values.get("topic"))
print()
print("Joke:\n", final_snapshot.values.get("joke"))
print()
print("Explanation:\n", final_snapshot.values.get("explanation"))

Topic:
 robots

Joke:
 Why did the robot get kicked out of the bakery? Because it kept making bread rolls—turns out, the Three Laws of Robotics don’t cover pastry dough! 🍞🤖

Explanation:
 The joke plays on the Three Laws of Robotics, which prioritize preventing harm to humans. The robot, following its programming, keeps making bread rolls (a literal "roll" of dough), but since the laws don’t address pastry dough mishaps, its overzealous baking causes a problem. The humor lies in the absurdity of applying strict robotic ethics to a mundane baking task, where the robot’s actions aren’t "harmful" but still disrupt the bakery.


## 12. Full State History

Shows every checkpoint saved — you can see the gap where the failure happened.

In [14]:
print("=== State History ===")
for snapshot in workflow.get_state_history(config):
    print(f"--- Step {snapshot.metadata['step']} | {snapshot.created_at} ---")
    print("Next:  ", snapshot.next)
    print("Values:", {k: v[:60] + "..." if isinstance(v, str) and len(v) > 60 else v
                     for k, v in snapshot.values.items()})
    print()

=== State History ===
--- Step 2 | 2026-06-17T11:13:05.201705+00:00 ---
Next:   ()
Values: {'topic': 'robots', 'joke': 'Why did the robot get kicked out of the bakery? Because it k...', 'explanation': 'The joke plays on the Three Laws of Robotics, which prioriti...'}

--- Step 1 | 2026-06-17T11:12:31.303252+00:00 ---
Next:   ('explain_joke',)
Values: {'topic': 'robots', 'joke': 'Why did the robot get kicked out of the bakery? Because it k...'}

--- Step 0 | 2026-06-17T11:12:29.322508+00:00 ---
Next:   ('generate_joke',)
Values: {'topic': 'robots'}

--- Step -1 | 2026-06-17T11:12:29.320313+00:00 ---
Next:   ('__start__',)
Values: {}

